# Notebook 7/10 — Geographic Audience Segmentation for Digital Marketing

**Use case:** enrich a CSV of website sessions with the **IP2Location.io** API and compare marketing activity by country, region, and acquisition channel.

**You will build:**
- A country-level audience report showing sessions, conversions, conversion rate, and revenue.
- Regional and channel-specific summaries plus quick visualizations.
- A cautious list of **markets to review** based on transparent, adjustable heuristics.
- Five CSV exports for downstream reporting.

**Important:** This is a *descriptive marketing analysis*, **not** a location-based identification, a statistically proven campaign lift, or an attribution/causal model. IP geolocation is approximate and is not GPS. A VPN, corporate proxy, or mobile connection may affect the results. The example input data, revenue, and demo geographic assignments are **synthetic** and not actual visitor traffic. The public IPs used in the sample are demonstrative service IP addresses, not actual website visitors; they may resolve to different locations in live mode.

API reference: https://www.ip2location.io/ip2location-documentation

## 1. Install dependencies

Run once if your Jupyter environment does not already have the libraries.

In [ ]:
%pip -q install pandas requests matplotlib

## 2. Imports and shared configuration

Run this cell **in both modes**. All shared constants are defined here, independently of the synthetic demo fixtures (to prevent `NameError` when switching to live mode).

- `MODE = "demo"`: **zero** network/API requests and clearly simulated locations.
- `MODE = "live"`: real HTTPS requests to IP2Location.io for unique public IPs.
- Place `sample_marketing_sessions.csv` next to the notebook, or change `INPUT_CSV`.
- Prefer the `IP2LOCATION_API_KEY` environment variable; do **not** commit API keys to notebooks.

In [ ]:
import ipaddress
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

MODE = "demo"  # change to "live" after reviewing the sample and API quota
API_URL = "https://api.ip2location.io/"
API_KEY = os.environ.get("IP2LOCATION_API_KEY", "").strip()
INPUT_CSV = Path("sample_marketing_sessions.csv")
OUTPUT_DIR = Path("outputs_notebook_07")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Prevent accidental large lookups; duplicate IPs are queried only once.
MAX_UNIQUE_IPS = 25
REQUEST_DELAY_SECONDS = 0.25
REQUEST_TIMEOUT_SECONDS = 12
MIN_COUNTRY_SESSIONS = 15
MIN_CONVERSIONS_FOR_EXPANSION = 3
EXPORT_RAW_IPS = False  # privacy-conscious default for row-level export

GEO_FIELDS = ["country_code", "country_name", "region_name", "city_name"]
REQUIRED_INPUT_COLUMNS = [
    "session_id", "event_time_utc", "ip", "channel", "campaign",
    "device", "converted", "revenue_usd"
]
def normalize_geography(ip, payload, status):
    """Return a consistent schema for available and unavailable location data."""
    result = {"ip": ip, "lookup_status": status, **{field: None for field in GEO_FIELDS}}
    if isinstance(payload, dict):
        for field in GEO_FIELDS:
            val = payload.get(field)
            result[field] = str(val).strip() if val not in (None, "", "-") else None
    return result

assert MODE in {"demo", "live"}, "MODE must be 'demo' or 'live'"
print(f"Mode: {MODE}; key configured: {bool(API_KEY)}; max unique IP lookups: {MAX_UNIQUE_IPS}")

## 3. Import session-level marketing data

Each CSV row is one website **session**. `converted` is `0` or `1`, and `revenue_usd` is a synthetic example amount in USD. Sessions, conversions, and revenue do **not** come from the IP2Location.io API — only geography does.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Missing {INPUT_CSV}. Download sample_marketing_sessions.csv "
        "and place it in the current Jupyter working directory."
    )
sessions = pd.read_csv(INPUT_CSV, dtype={"ip": "string", "session_id": "string"}, keep_default_na=False)
missing = sorted(set(REQUIRED_INPUT_COLUMNS) - set(sessions.columns))
if missing:
    raise ValueError(f"Missing required input columns: {missing}")
if sessions["session_id"].duplicated().any():
    raise ValueError("session_id must uniquely identify a session")
sessions["event_time_utc"] = pd.to_datetime(sessions["event_time_utc"], utc=True, errors="coerce")
if sessions["event_time_utc"].isna().any():
    raise ValueError("Some event_time_utc timestamps could not be parsed")
sessions["converted"] = pd.to_numeric(sessions["converted"], errors="raise")
if not sessions["converted"].isin([0, 1]).all():
    raise ValueError("converted must contain only 0 or 1")
sessions["converted"] = sessions["converted"].astype(int)
sessions["revenue_usd"] = pd.to_numeric(sessions["revenue_usd"], errors="raise")
if sessions["revenue_usd"].isna().any() or (sessions["revenue_usd"] < 0).any():
    raise ValueError("revenue_usd must be a nonnegative amount")
print("Sessions loaded:", len(sessions))
display(sessions.head())

## 4. Validate public IPs and estimate request usage

Private, loopback, reserved, missing, and malformed addresses are **not** sent to the API. Valid public IPs are deduplicated; a thousand sessions sharing one IP require only one API lookup.

In [ ]:
def ip_status(value):
    raw = str(value).strip()
    if not raw:
        return "missing"
    try:
        return "public" if ipaddress.ip_address(raw).is_global else "non_public"
    except ValueError:
        return "invalid"

sessions["ip"] = sessions["ip"].astype(str).str.strip()
sessions["ip_status"] = sessions["ip"].map(ip_status)
unique_public_ips = sorted(sessions.loc[sessions["ip_status"] == "public", "ip"].unique().tolist())
print("IP validation counts:")
display(sessions["ip_status"].value_counts().rename_axis("ip_status").to_frame("sessions"))
print(f"Distinct public IPs: {len(unique_public_ips)}")
print(f"At most {min(len(unique_public_ips), MAX_UNIQUE_IPS)} IP lookups in {MODE!r} mode")
if len(unique_public_ips) > MAX_UNIQUE_IPS:
    print("NOTE: Some public IPs will be skipped because MAX_UNIQUE_IPS is reached.")

## 5. Demonstration fixtures (offline only)

This dictionary assigns **synthetic** locations to the sample IPs to make the marketing summaries interesting offline. These are **not assertions** about the IP addresses' actual locations. When `MODE = "live"`, *none of these fixtures are used*.

In [ ]:
DEMO_GEO = {
    "8.8.8.8": ("US", "United States", "California", "Mountain View"),
    "208.67.222.222": ("US", "United States", "California", "San Francisco"),
    "1.9.1.9": ("MY", "Malaysia", "Kuala Lumpur", "Kuala Lumpur"),
    "202.188.0.133": ("MY", "Malaysia", "Selangor", "Shah Alam"),
    "165.21.83.88": ("SG", "Singapore", "Central Singapore", "Singapore"),
    "203.116.1.78": ("SG", "Singapore", "Central Singapore", "Singapore"),
    "194.150.168.168": ("DE", "Germany", "Berlin", "Berlin"),
    "80.156.86.78": ("DE", "Germany", "Bavaria", "Munich"),
    "210.130.1.1": ("JP", "Japan", "Tokyo", "Tokyo"),
    "202.12.27.33": ("JP", "Japan", "Osaka", "Osaka"),
    "200.160.0.8": ("BR", "Brazil", "São Paulo", "São Paulo"),
    "200.160.2.3": ("BR", "Brazil", "Rio de Janeiro", "Rio de Janeiro"),
}

def demo_lookup(ip):
    geo = DEMO_GEO.get(ip)
    if not geo:
        return normalize_geography(ip, None, "demo_unmapped")
    return normalize_geography(ip, dict(zip(GEO_FIELDS, geo)), "demo_synthetic")

print(f"Synthetic demo mappings available: {len(DEMO_GEO)}")

## 6. Live API helper

Uses the officially documented endpoint and optional `Authorization: Bearer` header. A request limit or authentication error **stops additional requests**; unavailable data are **not** recorded as actual countries. No automatic high-volume retry loop is used.

**Tip:** For your own production use, implement privacy notice/retention safeguards, appropriate rate limiting, and persistent caching to reduce repeat charges. This notebook intentionally keeps caching to the current run only.

In [ ]:
def live_lookup(ip, client):
    headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
    try:
        response = client.get(
            API_URL,
            params={"ip": ip, "format": "json"},
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        if response.status_code == 429:
            return normalize_geography(ip, None, "rate_limited"), True
        if response.status_code in (401, 403):
            return normalize_geography(ip, None, "authentication_or_access_error"), True
        response.raise_for_status()
        data = response.json()
        if not isinstance(data, dict):
            return normalize_geography(ip, None, "invalid_response"), False
        if data.get("error") or data.get("error_message"):
            return normalize_geography(ip, None, "api_error"), True
        if not data.get("country_code"):
            return normalize_geography(ip, None, "missing_geography"), False
        return normalize_geography(ip, data, "live_ok"), False
    except (requests.RequestException, ValueError) as exc:
        print(f"Warning: lookup failed for {ip}: {type(exc).__name__}")
        return normalize_geography(ip, None, "request_failed"), False

print("API helper ready. No request has been made by this cell.")

## 7. Geolocate each unique public IP, then enrich sessions

In demo mode, there are no network calls. In live mode, only publicly routable addresses are queried and the maximum unique-IP limit is enforced. Demo locations are never used as a live fallback.

In [ ]:
geo_results = []
stop_requests = False
with requests.Session() as client:
    for idx, ip in enumerate(unique_public_ips):
        if idx >= MAX_UNIQUE_IPS:
            geo_results.append(normalize_geography(ip, None, "skipped_max_unique_ips"))
            continue
        if MODE == "demo":
            geo_results.append(demo_lookup(ip))
        elif stop_requests:
            geo_results.append(normalize_geography(ip, None, "skipped_after_api_limit"))
        else:
            result, stop_requests = live_lookup(ip, client)
            geo_results.append(result)
            if not stop_requests and idx + 1 < min(len(unique_public_ips), MAX_UNIQUE_IPS):
                time.sleep(REQUEST_DELAY_SECONDS)

geolocation = pd.DataFrame(geo_results, columns=["ip", "lookup_status"] + GEO_FIELDS)
# Even an empty input retains the required schema.
enriched = sessions.merge(geolocation, on="ip", how="left", validate="many_to_one")
enriched["lookup_status"] = enriched["lookup_status"].fillna("not_queried_non_public")
enriched["country_code"] = enriched["country_code"].fillna("ZZ")
enriched["country_name"] = enriched["country_name"].fillna("Unresolved")
enriched["region_name"] = enriched["region_name"].fillna("Unknown region")
enriched["city_name"] = enriched["city_name"].fillna("Unknown city")
enriched["region_market"] = enriched["country_code"] + " / " + enriched["region_name"]
print("Lookup results (unique IPs):")
display(geolocation["lookup_status"].value_counts().rename_axis("status").to_frame("ips"))
print("Enriched session preview:")
display(enriched[["session_id", "ip_status", "lookup_status", "country_name", "region_name", "channel", "converted", "revenue_usd"]].head(8))

## 8. Aggregate geographic marketing metrics

- **Sessions:** input rows, not unique people.
- **Conversions:** total `converted == 1` sessions.
- **Conversion rate:** conversions / sessions × 100.
- **Revenue per session:** total sample USD revenue / sessions.
- **Unique IPs:** deduplicated network addresses, not individual visitors.

`Unresolved` rows are reported for data-quality transparency but **not used for country recommendations**.

In [ ]:
def make_kpis(frame, group_cols):
    result = frame.groupby(group_cols, dropna=False).agg(
        sessions=("session_id", "size"),
        conversions=("converted", "sum"),
        revenue_usd=("revenue_usd", "sum"),
        unique_ips=("ip", "nunique"),
    ).reset_index()
    result["conversion_rate_pct"] = (100 * result["conversions"] / result["sessions"]).round(2)
    result["revenue_per_session_usd"] = (result["revenue_usd"] / result["sessions"]).round(2)
    result["avg_order_value_usd"] = (
        result["revenue_usd"] / result["conversions"].replace(0, float("nan"))
    ).round(2)
    result["revenue_usd"] = result["revenue_usd"].round(2)
    return result.sort_values(["sessions", "revenue_usd"], ascending=[False, False]).reset_index(drop=True)

country_kpis = make_kpis(enriched, ["country_code", "country_name"])
region_kpis = make_kpis(enriched.loc[enriched["country_code"] != "ZZ"],
                        ["country_code", "country_name", "region_name"])
channel_kpis = make_kpis(enriched, ["country_code", "country_name", "channel"])
print("Country-level performance (includes unresolved geography):")
display(country_kpis)
print("Regions (preview):")
display(region_kpis.head(10))

## 9. Visualize geographic demand and conversion

Simple plots, without mapping exact visitor coordinates. Compare rates only when sample sizes are meaningful for the business question.

In [ ]:
known_countries = country_kpis.loc[country_kpis["country_code"] != "ZZ"].copy()
if not known_countries.empty:
    plt.figure(figsize=(10, 4))
    plt.bar(known_countries["country_name"], known_countries["sessions"])
    plt.title(f"Website sessions by country — {MODE} mode")
    plt.ylabel("Sessions")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

    qualified_plot = known_countries.loc[known_countries["sessions"] >= MIN_COUNTRY_SESSIONS]
    if not qualified_plot.empty:
        plt.figure(figsize=(10, 4))
        plt.bar(qualified_plot["country_name"], qualified_plot["conversion_rate_pct"])
        plt.title(f"Observed conversion rate by country (≥{MIN_COUNTRY_SESSIONS} sessions)")
        plt.ylabel("Conversion rate (%)")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough per-country sessions for a conversion-rate comparison.")
else:
    print("No resolved countries available to plot.")

## 10. Cross-tab acquisition channels by geographic market

Channel-level segmentation helps identify which sources bring sessions to each country. This chart shows **volume**, not incremental campaign lift.

In [ ]:
channel_mix = pd.crosstab(
    enriched.loc[enriched["country_code"] != "ZZ", "country_name"],
    enriched.loc[enriched["country_code"] != "ZZ", "channel"],
)
if not channel_mix.empty:
    display(channel_mix)
    ax = channel_mix.plot(kind="bar", stacked=True, figsize=(11, 5))
    ax.set_title(f"Session acquisition channels by country — {MODE} mode")
    ax.set_xlabel("Country")
    ax.set_ylabel("Sessions")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No resolved countries available for channel segmentation.")

## 11. Create a *review queue* for geographic marketing opportunities

These labels are **rules of thumb, not statistically significant findings**. The baseline uses only sessions with resolved geography. Users should check sample size, acquisition mix, campaign costs, attribution windows, seasonality, and privacy obligations before any targeting decision.

The rules are intentionally conservative: too-small samples are flagged as needing more data; high conversion rate alone does not prove that raising ad spend will increase profit.

In [ ]:
known = enriched.loc[enriched["country_code"] != "ZZ"]
base_rate = (100 * known["converted"].mean()) if len(known) else float("nan")

def market_review(row):
    if row["sessions"] < MIN_COUNTRY_SESSIONS:
        return "Insufficient sample — gather more data"
    if pd.isna(base_rate) or base_rate == 0:
        return "Review manually — baseline unavailable"
    if row["conversions"] >= MIN_CONVERSIONS_FOR_EXPANSION and row["conversion_rate_pct"] >= base_rate * 1.20:
        return "Review for a controlled expansion test"
    if row["conversion_rate_pct"] <= base_rate * 0.80:
        return "Review landing page, offer, and acquisition mix"
    return "Monitor performance and collect more data"

market_reviews = known_countries.copy()
market_reviews["review_suggestion"] = market_reviews.apply(market_review, axis=1)
market_reviews["observed_baseline_conversion_pct"] = round(base_rate, 2) if pd.notna(base_rate) else None
print(f"Resolved-geography baseline conversion rate: {base_rate:.2f}%" if pd.notna(base_rate) else "No resolved geography")
display(market_reviews[["country_name", "sessions", "conversions", "conversion_rate_pct", "review_suggestion"]])

## 12. Data-quality and export checks

The default enriched-session export **omits raw IP addresses**. You can opt in when appropriate, but real session IPs and account-level activity deserve careful handling under your organization's privacy rules.

The aggregate CSVs contain country/region-level metrics only. When reporting, always label demo output as synthetic.

In [ ]:
print("Mode:", MODE)
print("Input sessions:", len(sessions))
print("Enriched sessions:", len(enriched))
assert len(enriched) == len(sessions), "Join must preserve all input sessions"
assert enriched["session_id"].is_unique, "Join unexpectedly duplicated rows"
assert int(country_kpis["sessions"].sum()) == len(sessions), "Country totals mismatch"
assert int(country_kpis["conversions"].sum()) == int(sessions["converted"].sum())
print("Lookup coverage by session:")
display(enriched["lookup_status"].value_counts().rename_axis("status").to_frame("sessions"))

session_export = enriched.copy()
if not EXPORT_RAW_IPS:
    session_export = session_export.drop(columns=["ip"])

outputs = {
    "marketing_sessions_enriched.csv": session_export,
    "marketing_country_kpis.csv": country_kpis,
    "marketing_region_kpis.csv": region_kpis,
    "marketing_channel_kpis.csv": channel_kpis,
    "marketing_market_review.csv": market_reviews,
}
for filename, frame in outputs.items():
    destination = OUTPUT_DIR / filename
    frame.to_csv(destination, index=False)
    print(f"Wrote {destination} ({len(frame)} rows)")

## 13. Apply the workflow to your own traffic

Replace `sample_marketing_sessions.csv` with your website analytics export. Provide one row per session and ensure it contains the required columns from Section 3. If you have more distinct IPs, increase `MAX_UNIQUE_IPS` deliberately within your account's quota; **this example is not the dedicated bulk API**.

**Real-world cautions:**
- The free keyless API documents a limited daily request allowance; authenticated accounts have their own plan limits. Check current plan details before increasing traffic.
- Some IPs represent a VPN, a carrier gateway, or a corporate network; geographic labels are approximate.
- Country-level conversion-rate differences can be driven by campaign mix or small samples. Test and validate decisions separately.
- Respect legal requirements for collecting, processing, and retaining IP addresses and session data.

**Next example (Notebook 8):** A time-zone-aware international campaign scheduling planner using geolocation plus configured market time zones.